In [1]:
# Let's implement a multi-agent system. 

In [2]:
import os
import requests
from dotenv import load_dotenv

In [3]:
from langchain_deepseek import ChatDeepSeek
from langchain_core import tools
from langchain.agents import create_agent
from langchain_tavily import TavilySearch, TavilyExtract
from tavily import TavilyClient
from langchain.tools import tool

In [4]:
load_dotenv()

True

In [5]:
personal_data = {
    "ineterests": [
        "studying in rainy weather under balcony", 
        "drinking tea and sweats",
        "walking and listening to music",
        "seeing happy people on street and passing their times",
        "early mornings in a cool weather",
        "afternoon, golden hour, purple sky",
        "street cats",
        "driving on a good road (road trip)"
    ],
    "deseases": ["high-functional autism of adults", "aura-migrane caused by harsh light"],
    "habits & diets": ["5000 steps per day", "8 hours study per day", "eating ice-cream and sewats"],
    "age": 23,
}

In [23]:
todo_prompt = f"I'm heading to a trip to Mahmood-Abad, Mazandaran acommidating in a cozy villa in a rich town with private access to sea-coast and close to woods, give me a list of tasks to do during next 5 days, knowing that {personal_data}"

In [26]:
weather_prompt = f"find the best weather for each of the given tasks and tell me why you chose that tasks to be done in that day according to the weather forecast."

In [7]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [8]:
tavily_tool = TavilySearch(
    max_results=5,
    topic="general",
    time_range="year"
)

In [9]:
extract_tool = TavilyExtract(
    extract_depth="advanced",  # "basic" or "advanced"
    format="markdown",          # "markdown" or "text"
    include_images=False,
)

In [10]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage
from langgraph.runtime import Runtime

In [11]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """Model-based guardrail: Use an LLM to evaluate response safety."""

    def __init__(self):
        super().__init__()
        # Use a fast/cheap model for the safety check
        self.safety_model = init_chat_model("deepseek-chat") 

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the final AI response
        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a model to evaluate safety
        safety_prompt = f"""Evaluate if this response is safe and appropriate.
        Respond with only 'SAFE' or 'UNSAFE'.

        Response: {last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            # Block/redact the unsafe response
            last_message.content = "I cannot provide that response. It violated our safety guidelines."
        
        return None

In [12]:
@tool("weather_tool", return_direct=True)
def weather_tool(city:str) -> dict:
    """returns the weather forecast for a given city"""
    weather_api_key = os.getenv("WEATHER_API_KEY")
    url = "http://api.weatherapi.com/v1/current.json"
    params = {
        "key": weather_api_key,
        "q": city
    }
    r = requests.get(url, params=params)

    if r.status_code == 200: 
        return r.json()
    else:
        return "I was desprete to fetch the weather data!"

In [14]:
# "deepseek-chat" is the deepseek model we're looking for, and now we're initializing the LLM model. 

In [15]:
llm = ChatDeepSeek(model = "deepseek-chat", temperature=0.7)

In [24]:
todo_agent = create_agent(
    model=llm,
    tools=[tavily_tool, extract_tool],
    system_prompt=todo_prompt, 
)

In [28]:
weather_agent = create_agent(
    model=llm, 
    tools=[weather_tool],
    system_prompt=weather_prompt
)